In [1]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

import torch

In [2]:
X, y = load_iris(return_X_y=True)

In [3]:
X.shape

(150, 4)

In [4]:
np.unique(y)

array([0, 1, 2])

In [5]:
class FCN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.fcn = torch.nn.Sequential(
            torch.nn.Linear(4, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 3)
        )

    def forward(self, X):
        X = self.fcn(X)
        return X

In [6]:
model = FCN()

In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [8]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.nn.functional.one_hot(torch.tensor(y)).to(dtype=torch.float32)

In [9]:
train_ds = torch.utils.data.TensorDataset(X_tensor, y_tensor)

In [10]:
mini_batch_size = 4

In [11]:
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False)

In [12]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [13]:
def fit(epochs, model, optimizer, train_dl):
    loss_func = torch.nn.CrossEntropyLoss()
    model.train()

    for epoch in range(epochs):
        for X_mb, y_mb in train_dl:
            y_hat = model(X_mb)
            cost = loss_func(y_hat, y_mb)
            cost.backward()
            optimizer.step()
            optimizer.zero_grad()

    return model

In [14]:
epochs = 100

In [15]:
fit(epochs, model, optimizer, train_dl)

FCN(
  (fcn): Sequential(
    (0): Linear(in_features=4, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=20, bias=True)
    (5): ReLU()
    (6): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [16]:
with torch.no_grad():
    yhat_train = model(train_ds[:][0])

In [17]:
yhat_train

tensor([[ 11.4743,   2.4984,  -9.7029],
        [  9.6280,   2.9629,  -9.0226],
        [ 11.4710,   2.6410,  -9.8469],
        [ 11.0325,   2.7520,  -9.6854],
        [ 12.2489,   2.3759, -10.0566],
        [ 11.3819,   2.3085,  -9.4563],
        [ 12.2727,   2.4725, -10.1710],
        [ 11.1761,   2.6089,  -9.6275],
        [ 10.8162,   2.7882,  -9.5928],
        [ 10.3508,   2.8540,  -9.3630],
        [ 11.3288,   2.4035,  -9.5153],
        [ 11.7149,   2.5536,  -9.9068],
        [ 10.3302,   2.8791,  -9.3777],
        [ 12.3974,   2.6046, -10.3884],
        [ 11.8292,   2.1561,  -9.5730],
        [ 12.9857,   1.9951, -10.1421],
        [ 11.8592,   2.2085,  -9.6487],
        [ 11.1392,   2.5109,  -9.5076],
        [ 10.2258,   2.4893,  -8.9157],
        [ 12.4240,   2.2772, -10.0686],
        [  9.5156,   2.8659,  -8.8475],
        [ 11.6391,   2.3451,  -9.6496],
        [ 13.9972,   2.2142, -10.9768],
        [  8.9671,   2.9426,  -8.5802],
        [ 11.3053,   2.6396,  -9.7359],


In [18]:
yhat_train = yhat_train.argmax(dim=1).numpy()

In [19]:
accuracy_score(y, yhat_train)

0.9866666666666667